# Final Gold Data Generation

This notebook creates the final datasets for visualization by combining all processed data from previous steps:
- Project-level data with occupations, skills, NACE codes, and O*NET education levels
- Clean, deduplicated datasets ready for analysis and visualization

## 0. Setup

In [1]:
import pandas as pd
import json
from pathlib import Path

# Data directories
DATA_DIR = Path("../data")
BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

# Project data paths
project_data_dir = BRONZE_DIR / "project_data"
all_project_details_path = project_data_dir / "all_project_details.csv"
project_summary_path = project_data_dir / "project_summary.csv"

# Silver data paths
short_summary_dir = SILVER_DIR / "short_summary_json"

print(f"Bronze data directory: {BRONZE_DIR}")
print(f"Silver data directory: {SILVER_DIR}")
print(f"Gold data directory: {GOLD_DIR}")
print(f"Project data directory: {project_data_dir}")

Bronze data directory: ../data/bronze
Silver data directory: ../data/silver
Gold data directory: ../data/gold
Project data directory: ../data/bronze/project_data


## 1. Get Project Details

### 1.01 Load Project Data

In [2]:
# Read project details and summary
all_project_details = pd.read_csv(all_project_details_path)
project_summary = pd.read_csv(project_summary_path)

print(f"All project details: {all_project_details.shape[0]} rows, {all_project_details.shape[1]} columns")
print(f"Project summary: {project_summary.shape[0]} rows, {project_summary.shape[1]} columns")

All project details: 123 rows, 20 columns
Project summary: 123 rows, 4 columns


### 1.02 Convert column names to snake_case

In [3]:
# Convert column names to snake_case
all_project_details.columns = (
    all_project_details.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace(r'[^\w]', '_', regex=True)
    .str.replace(r'_+', '_', regex=True)
    .str.strip('_')
)

print("Column names after standardization:")
print(list(all_project_details.columns))

Column names after standardization:
['project_id', 'status', 'team_leader', 'borrower_2', 'country', 'disclosure_date', 'approval_date', 'effective_date', 'total_project_cost_1', 'implementing_agency', 'region', 'fiscal_year_3', 'commitment_amount', 'environmental_category', 'environmental_and_social_risk', 'closing_date', 'last_stage_reached', 'last_update_date', 'consultant_services_required', 'associated_projects']


### 1.03 Merge and Filter Projects

In [4]:
# Merge project_summary to all_project_details
all_projects = all_project_details.merge(
    project_summary,
    on="project_id",
    how="left"
)

print(f"Merged data: {all_projects.shape[0]} rows, {all_projects.shape[1]} columns")

# Keep only projects with downloaded PADs
all_projects = all_projects[all_projects["pads_downloaded"] >= 1]

print(f"Projects with downloaded PADs: {all_projects.shape[0]} rows")
print(f"\nFirst few projects:")
print(all_projects.head())

Merged data: 123 rows, 23 columns
Projects with downloaded PADs: 98 rows

First few projects:
  project_id  status                                        team_leader  \
1    P119893  Closed                    Abdulhakim Mohammed Abdisubhan    
3    P173506  Active  Didier Makoso Tsasa , Fabrice Karl Bertholet, ...   
4    P176731  Active  Janina Franco , Abdulhakim Mohammed Abdisubhan...   
5    P507759  Active                     Jenny Jing Chao , Maria Arango   
6    P180547  Active   Monali Ranade , Dana Rysankova, Alona Kazantseva   

                                          borrower_2  \
1            Federal Democratic Republic of Ethiopia   
3                       DEMOCRATIC REPUBLIC OF CONGO   
4            Federal Democratic Republic of Ethiopia   
5                             Republic of Mozambique   
6  Common Market for Eastern and Southern Africa ...   

                         country    disclosure_date  \
1                       Ethiopia  December 22, 2011   
3  Congo

## 2. Select Projects

### 2.01 Configure Selection Mode

In [5]:
# ========================================
# PROJECT SELECTION MODE
# ========================================
# Options: "custom", "first_n", "all"
SELECTION_MODE = "custom"

# If "custom": define your list here
# Auto-populate from unique_esco_nace_onet_csv directory
# CUSTOM_PROJECTS = ['P119893', 'P173506', 'P176731', 'P507759', 'P180547']
unique_esco_nace_onet_dir = SILVER_DIR / "unique_esco_nace_onet_csv"
CUSTOM_PROJECTS = sorted([
    f.stem.replace('_esco_nace_onet', '') 
    for f in unique_esco_nace_onet_dir.glob('*_esco_nace_onet.csv')
])

# If "first_n": specify how many
FIRST_N = 50

# ========================================

print(f"Selection mode: {SELECTION_MODE}")
if SELECTION_MODE == "custom":
    print(f"Custom projects count: {len(CUSTOM_PROJECTS)}")
elif SELECTION_MODE == "first_n":
    print(f"First N projects: {FIRST_N}")

Selection mode: custom
Custom projects count: 98


### 2.02 Apply Selection and Filter Projects

In [6]:
# Apply selection based on mode
if SELECTION_MODE == "custom":
    selected_projects = CUSTOM_PROJECTS
elif SELECTION_MODE == "first_n":
    selected_projects = all_projects["project_id"][:FIRST_N].tolist()
elif SELECTION_MODE == "all":
    selected_projects = all_projects["project_id"].tolist()
else:
    raise ValueError(f"Invalid SELECTION_MODE: {SELECTION_MODE}")

print(f"Selected {len(selected_projects)} projects")

# Filter projects dataframe for selected projects
projects_df = all_projects[all_projects["project_id"].isin(selected_projects)]

print(f"\nFiltered to {len(projects_df)} projects")
print(f"Sample projects:")
print(projects_df[["project_id", "status", "country"]].head(10).to_string(index=False))

Selected 98 projects

Filtered to 98 projects
Sample projects:
project_id status                       country
   P119893 Closed                      Ethiopia
   P173506 Active Congo, Democratic Republic of
   P176731 Active                      Ethiopia
   P507759 Active                    Mozambique
   P180547 Active   Eastern and Southern Africa
   P505856 Active                    Seychelles
   P181341 Active  Somalia, Federal Republic of
   P075941 Closed   Eastern and Southern Africa
   P160708 Active    Western and Central Africa
   P153743 Closed                         Niger


### 2.03 Save Selected Projects Data

In [7]:
# Save filtered projects to silver directory
output_dir = SILVER_DIR / "selected_projects_data"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "selected_projects.csv"
projects_df.to_csv(output_path, index=False)

print(f"Saved {len(projects_df)} projects to:")
print(f"  {output_path}")

Saved 98 projects to:
  ../data/silver/selected_projects_data/selected_projects.csv


## 3. Project Level Data

### 3.01 Load project data

In [8]:
# Verify selected projects data (already created in Section 2)
print(f"Working with {len(projects_df)} projects")
print(f"\nColumns: {list(projects_df.columns)}")
projects_df.head()

Working with 98 projects

Columns: ['project_id', 'status', 'team_leader', 'borrower_2', 'country', 'disclosure_date', 'approval_date', 'effective_date', 'total_project_cost_1', 'implementing_agency', 'region', 'fiscal_year_3', 'commitment_amount', 'environmental_category', 'environmental_and_social_risk', 'closing_date', 'last_stage_reached', 'last_update_date', 'consultant_services_required', 'associated_projects', 'details_found', 'pads_found', 'pads_downloaded']


,project_id,status,team_leader,borrower_2,country,disclosure_date,approval_date,effective_date,total_project_cost_1,implementing_agency,...,environmental_category,environmental_and_social_risk,closing_date,last_stage_reached,last_update_date,consultant_services_required,associated_projects,details_found,pads_found,pads_downloaded
1,P119893,Closed,Abdulhakim Mohammed Abdisubhan,Federal Democratic Republic of Ethiopia,Ethiopia,"December 22, 2011",(as of board presentation),"January 4, 2013",US$ 275.00 million,"Development Bank of Ethiopia,Ethiopia Electric...",...,B,Not Applicable,"March 31, 2025",Bank Approved,"June 20, 2024",No,P155563,1,1,1
3,P173506,Active,"Didier Makoso Tsasa , Fabrice Karl Bertholet, ...",DEMOCRATIC REPUBLIC OF CONGO,"Congo, Democratic Republic of","January 6, 2021",(as of board presentation),"May 18, 2023",US$ 939.00 million,Ministère des Ressources Hydrauliques et de l'...,...,NaN,High,"September 30, 2029",Bank Approved,"June 17, 2024",Yes,NaN,1,1,1
4,P176731,Active,"Janina Franco , Abdulhakim Mohammed Abdisubhan...",Federal Democratic Republic of Ethiopia,Ethiopia,"August 15, 2023",(as of board presentation),"June 19, 2024",US$ 537.00 million,"Ethiopia Electric Power,Ethiopia Electric Utility",...,NaN,High,"August 30, 2030",Bank Approved,"June 16, 2023",No,NaN,1,1,1
5,P507759,Active,"Jenny Jing Chao , Maria Arango",Republic of Mozambique,Mozambique,"November 29, 2024",(as of board presentation),"August 21, 2025",US$ 0.00 million,"Electricidade de Moçambique (EDM),Fundo de Ene...",...,NaN,Substantial,"December 31, 2030",Bank Approved,"March 22, 2025",TBD,P512418,1,1,1
6,P180547,Active,"Monali Ranade , Dana Rysankova, Alona Kazantseva",Common Market for Eastern and Southern Africa ...,Eastern and Southern Africa,"July 7, 2023",(as of board presentation),"April 5, 2024",US$ 10000.00 million,Common Market for Eastern and Southern Africa ...,...,NaN,Substantial,"December 31, 2030",Bank Approved,"July 16, 2023",Yes,NaN,1,1,1


In [9]:
# Keep only relevant columns
columns_to_keep = [
    'project_id', 
    'status', 
    'team_leader', 
    'borrower_2', 
    'country', 
    'disclosure_date', 
    'effective_date', 
    'total_project_cost_1', 
    'implementing_agency', 
    'region', 
    'environmental_and_social_risk', 
    'closing_date', 
    'last_stage_reached', 
    'last_update_date'
]

projects_df = projects_df[columns_to_keep]
print(f"Filtered to {len(columns_to_keep)} columns")
projects_df.head()

Filtered to 14 columns


,project_id,status,team_leader,borrower_2,country,disclosure_date,effective_date,total_project_cost_1,implementing_agency,region,environmental_and_social_risk,closing_date,last_stage_reached,last_update_date
1,P119893,Closed,Abdulhakim Mohammed Abdisubhan,Federal Democratic Republic of Ethiopia,Ethiopia,"December 22, 2011","January 4, 2013",US$ 275.00 million,"Development Bank of Ethiopia,Ethiopia Electric...",Eastern and Southern Africa,Not Applicable,"March 31, 2025",Bank Approved,"June 20, 2024"
3,P173506,Active,"Didier Makoso Tsasa , Fabrice Karl Bertholet, ...",DEMOCRATIC REPUBLIC OF CONGO,"Congo, Democratic Republic of","January 6, 2021","May 18, 2023",US$ 939.00 million,Ministère des Ressources Hydrauliques et de l'...,Eastern and Southern Africa,High,"September 30, 2029",Bank Approved,"June 17, 2024"
4,P176731,Active,"Janina Franco , Abdulhakim Mohammed Abdisubhan...",Federal Democratic Republic of Ethiopia,Ethiopia,"August 15, 2023","June 19, 2024",US$ 537.00 million,"Ethiopia Electric Power,Ethiopia Electric Utility",Eastern and Southern Africa,High,"August 30, 2030",Bank Approved,"June 16, 2023"
5,P507759,Active,"Jenny Jing Chao , Maria Arango",Republic of Mozambique,Mozambique,"November 29, 2024","August 21, 2025",US$ 0.00 million,"Electricidade de Moçambique (EDM),Fundo de Ene...",Eastern and Southern Africa,Substantial,"December 31, 2030",Bank Approved,"March 22, 2025"
6,P180547,Active,"Monali Ranade , Dana Rysankova, Alona Kazantseva",Common Market for Eastern and Southern Africa ...,Eastern and Southern Africa,"July 7, 2023","April 5, 2024",US$ 10000.00 million,Common Market for Eastern and Southern Africa ...,Eastern and Southern Africa,Substantial,"December 31, 2030",Bank Approved,"July 16, 2023"


### 3.02 Merge long summaries

In [10]:
# Read all long summary files from pad_summaries directory
summaries_dir = SILVER_DIR / "pad_summaries"
summaries = []

for summary_file in summaries_dir.glob("*_summary.txt"):
    project_id = summary_file.stem.replace("_summary", "")
    with open(summary_file, "r", encoding="utf-8") as f:
        summary_text = f.read()
    summaries.append({
        "project_id": project_id,
        "long_summary": summary_text
    })

summaries_df = pd.DataFrame(summaries)
print(f"Loaded {len(summaries_df)} long summaries")

# Merge with projects data
projects_df = projects_df.merge(summaries_df, on="project_id", how="left")
print(f"\nMerged dataframe shape: {projects_df.shape}")
print(f"Projects with summaries: {projects_df['long_summary'].notna().sum()}")
projects_df.head()

Loaded 98 long summaries

Merged dataframe shape: (98, 15)
Projects with summaries: 98


,project_id,status,team_leader,borrower_2,country,disclosure_date,effective_date,total_project_cost_1,implementing_agency,region,environmental_and_social_risk,closing_date,last_stage_reached,last_update_date,long_summary
0,P119893,Closed,Abdulhakim Mohammed Abdisubhan,Federal Democratic Republic of Ethiopia,Ethiopia,"December 22, 2011","January 4, 2013",US$ 275.00 million,"Development Bank of Ethiopia,Ethiopia Electric...",Eastern and Southern Africa,Not Applicable,"March 31, 2025",Bank Approved,"June 20, 2024",The Electricity Network Reinforcement and Expa...
1,P173506,Active,"Didier Makoso Tsasa , Fabrice Karl Bertholet, ...",DEMOCRATIC REPUBLIC OF CONGO,"Congo, Democratic Republic of","January 6, 2021","May 18, 2023",US$ 939.00 million,Ministère des Ressources Hydrauliques et de l'...,Eastern and Southern Africa,High,"September 30, 2029",Bank Approved,"June 17, 2024",The AGREE project is a large water and energy ...
2,P176731,Active,"Janina Franco , Abdulhakim Mohammed Abdisubhan...",Federal Democratic Republic of Ethiopia,Ethiopia,"August 15, 2023","June 19, 2024",US$ 537.00 million,"Ethiopia Electric Power,Ethiopia Electric Utility",Eastern and Southern Africa,High,"August 30, 2030",Bank Approved,"June 16, 2023",This project supports a phased electricity pro...
3,P507759,Active,"Jenny Jing Chao , Maria Arango",Republic of Mozambique,Mozambique,"November 29, 2024","August 21, 2025",US$ 0.00 million,"Electricidade de Moçambique (EDM),Fundo de Ene...",Eastern and Southern Africa,Substantial,"December 31, 2030",Bank Approved,"March 22, 2025",ASCENT Mozambique is Phase 10 of the ASCENT mu...
4,P180547,Active,"Monali Ranade , Dana Rysankova, Alona Kazantseva",Common Market for Eastern and Southern Africa ...,Eastern and Southern Africa,"July 7, 2023","April 5, 2024",US$ 10000.00 million,Common Market for Eastern and Southern Africa ...,Eastern and Southern Africa,Substantial,"December 31, 2030",Bank Approved,"July 16, 2023",ASCENT is a regional program to accelerate sus...


### 3.03 Load short summaries and geography

In [11]:
# Load all short summary JSON files and populate columns
short_summaries = []
geographies = []

for idx, row in projects_df.iterrows():
    project_id = row['project_id']
    json_file = short_summary_dir / f"{project_id}.json"
    
    if json_file.exists():
        with open(json_file, "r", encoding="utf-8") as f:
            data = json.load(f)
            short_summaries.append(data.get("summary", ""))
            geographies.append(data.get("geographic_scope", ""))
    else:
        short_summaries.append(None)
        geographies.append(None)

# Add new columns to dataframe
projects_df['short_summary'] = short_summaries
projects_df['geography'] = geographies

print(f"Added columns: short_summary, geography")
print(f"Projects with short summaries: {projects_df['short_summary'].notna().sum()}")
print(f"Projects with geography: {projects_df['geography'].notna().sum()}")
projects_df.head()

Added columns: short_summary, geography
Projects with short summaries: 98
Projects with geography: 98


,project_id,status,team_leader,borrower_2,country,disclosure_date,effective_date,total_project_cost_1,implementing_agency,region,environmental_and_social_risk,closing_date,last_stage_reached,last_update_date,long_summary,short_summary,geography
0,P119893,Closed,Abdulhakim Mohammed Abdisubhan,Federal Democratic Republic of Ethiopia,Ethiopia,"December 22, 2011","January 4, 2013",US$ 275.00 million,"Development Bank of Ethiopia,Ethiopia Electric...",Eastern and Southern Africa,Not Applicable,"March 31, 2025",Bank Approved,"June 20, 2024",The Electricity Network Reinforcement and Expa...,The Electricity Network Reinforcement and Expa...,Ethiopia
1,P173506,Active,"Didier Makoso Tsasa , Fabrice Karl Bertholet, ...",DEMOCRATIC REPUBLIC OF CONGO,"Congo, Democratic Republic of","January 6, 2021","May 18, 2023",US$ 939.00 million,Ministère des Ressources Hydrauliques et de l'...,Eastern and Southern Africa,High,"September 30, 2029",Bank Approved,"June 17, 2024",The AGREE project is a large water and energy ...,AGREE project expands renewable-based electric...,Democratic Republic of Congo
2,P176731,Active,"Janina Franco , Abdulhakim Mohammed Abdisubhan...",Federal Democratic Republic of Ethiopia,Ethiopia,"August 15, 2023","June 19, 2024",US$ 537.00 million,"Ethiopia Electric Power,Ethiopia Electric Utility",Eastern and Southern Africa,High,"August 30, 2030",Bank Approved,"June 16, 2023",This project supports a phased electricity pro...,This project supports Ethiopia's phased electr...,Ethiopia
3,P507759,Active,"Jenny Jing Chao , Maria Arango",Republic of Mozambique,Mozambique,"November 29, 2024","August 21, 2025",US$ 0.00 million,"Electricidade de Moçambique (EDM),Fundo de Ene...",Eastern and Southern Africa,Substantial,"December 31, 2030",Bank Approved,"March 22, 2025",ASCENT Mozambique is Phase 10 of the ASCENT mu...,"ASCENT Mozambique finances on-grid expansion, ...",Mozambique
4,P180547,Active,"Monali Ranade , Dana Rysankova, Alona Kazantseva",Common Market for Eastern and Southern Africa ...,Eastern and Southern Africa,"July 7, 2023","April 5, 2024",US$ 10000.00 million,Common Market for Eastern and Southern Africa ...,Eastern and Southern Africa,Substantial,"December 31, 2030",Bank Approved,"July 16, 2023",ASCENT is a regional program to accelerate sus...,ASCENT accelerates sustainable and clean energ...,Eastern and Southern Africa (more than 20 Coun...


### 1.04 Merge project titles

In [12]:
# Load project titles from bronze data
bronze_data_file = DATA_DIR / "bronze" / "project_data" / "all_pads_only.csv"
titles_df = pd.read_csv(bronze_data_file)

# Select only project_id (pid) and doc_title columns
titles_df = titles_df[['pid', 'doc_title']].rename(columns={'pid': 'project_id'})

# Remove " (English)" from titles
titles_df['doc_title'] = titles_df['doc_title'].str.replace(' (English)', '', regex=False)

# Rename doc_title to project_title
titles_df = titles_df.rename(columns={'doc_title': 'project_title'})

print(f"Loaded {len(titles_df)} project titles")

# Merge with projects data
projects_df = projects_df.merge(titles_df, on='project_id', how='left')

print(f"Merged dataframe shape: {projects_df.shape}")
print(f"Projects with titles: {projects_df['project_title'].notna().sum()}")


Loaded 98 project titles
Merged dataframe shape: (98, 18)
Projects with titles: 98


In [13]:
projects_df.head(11)

,project_id,status,team_leader,borrower_2,country,disclosure_date,effective_date,total_project_cost_1,implementing_agency,region,environmental_and_social_risk,closing_date,last_stage_reached,last_update_date,long_summary,short_summary,geography,project_title
0,P119893,Closed,Abdulhakim Mohammed Abdisubhan,Federal Democratic Republic of Ethiopia,Ethiopia,"December 22, 2011","January 4, 2013",US$ 275.00 million,"Development Bank of Ethiopia,Ethiopia Electric...",Eastern and Southern Africa,Not Applicable,"March 31, 2025",Bank Approved,"June 20, 2024",The Electricity Network Reinforcement and Expa...,The Electricity Network Reinforcement and Expa...,Ethiopia,Ethiopia - Electricity Network Reinforcement a...
1,P173506,Active,"Didier Makoso Tsasa , Fabrice Karl Bertholet, ...",DEMOCRATIC REPUBLIC OF CONGO,"Congo, Democratic Republic of","January 6, 2021","May 18, 2023",US$ 939.00 million,Ministère des Ressources Hydrauliques et de l'...,Eastern and Southern Africa,High,"September 30, 2029",Bank Approved,"June 17, 2024",The AGREE project is a large water and energy ...,AGREE project expands renewable-based electric...,Democratic Republic of Congo,"Congo, Democratic Republic of - Access Governa..."
2,P176731,Active,"Janina Franco , Abdulhakim Mohammed Abdisubhan...",Federal Democratic Republic of Ethiopia,Ethiopia,"August 15, 2023","June 19, 2024",US$ 537.00 million,"Ethiopia Electric Power,Ethiopia Electric Utility",Eastern and Southern Africa,High,"August 30, 2030",Bank Approved,"June 16, 2023",This project supports a phased electricity pro...,This project supports Ethiopia's phased electr...,Ethiopia,"Ethiopia - Power Sector Reform, Investment, an..."
3,P507759,Active,"Jenny Jing Chao , Maria Arango",Republic of Mozambique,Mozambique,"November 29, 2024","August 21, 2025",US$ 0.00 million,"Electricidade de Moçambique (EDM),Fundo de Ene...",Eastern and Southern Africa,Substantial,"December 31, 2030",Bank Approved,"March 22, 2025",ASCENT Mozambique is Phase 10 of the ASCENT mu...,"ASCENT Mozambique finances on-grid expansion, ...",Mozambique,Mozambique - Accelerating Sustainable and Clea...
4,P180547,Active,"Monali Ranade , Dana Rysankova, Alona Kazantseva",Common Market for Eastern and Southern Africa ...,Eastern and Southern Africa,"July 7, 2023","April 5, 2024",US$ 10000.00 million,Common Market for Eastern and Southern Africa ...,Eastern and Southern Africa,Substantial,"December 31, 2030",Bank Approved,"July 16, 2023",ASCENT is a regional program to accelerate sus...,ASCENT accelerates sustainable and clean energ...,Eastern and Southern Africa (more than 20 Coun...,Eastern and Southern Africa - Accelerating Sus...
5,P505856,Active,"Aalok Raj Pandey , Amit Jain","Public Utilities Corporation,The Republic of S...",Seychelles,"November 25, 2024",NaN,US$ 0.00 million,"Ministry of Agriculture, Climate Change, and E...",Eastern and Southern Africa,Moderate,"May 31, 2030",Bank Approved,"April 14, 2025",The Renewable Energy Acceleration Program Mult...,The Renewable Energy Acceleration Program Mult...,Seychelles,Seychelles - Renewable Energy Acceleration Pro...
6,P181341,Active,"Tigran Parvanyan , Paul Baringanire",Federal Ministry of Finance Somalia,"Somalia, Federal Republic of","September 18, 2023","March 14, 2024",US$ 0.00 million,Ministry of Energy and Water Resources,Eastern and Southern Africa,Substantial,"November 30, 2028",Bank Approved,"July 13, 2023",This PAD presents the ASCENT program for Easte...,ASCENT finances regional and national platform...,Somalia,Eastern and Southern Africa - Accelerating Sus...
7,P075941,Closed,"Norah Kipwola , Laurencia Karimi Njagi","Republic of Burundi,Republic of Rwanda,United ...",Eastern and Southern Africa,"February 3, 2011","October 24, 2006",US$ 468.90 million,"NBI NELSAP Coordination Unit (NELSAP-CU),Rusum...",Eastern and Southern Africa,Not Applicable,"June 30, 2025",Bank Approved,"May 4, 2024",The Rusumo Falls Hydroelectric Project is a re...,The Rusumo Falls Hydroelectric Project financ

In [14]:
projects_df.columns

Index(['project_id', 'status', 'team_leader', 'borrower_2', 'country',
       'disclosure_date', 'effective_date', 'total_project_cost_1',
       'implementing_agency', 'region', 'environmental_and_social_risk',
       'closing_date', 'last_stage_reached', 'last_update_date',
       'long_summary', 'short_summary', 'geography', 'project_title'],
      dtype='object')

### 1.05 Reorder columns

In [15]:
# Reorder columns (dropping country)
column_order = [
    'project_id', 'project_title', 'geography', 'short_summary', 'long_summary', 
    'closing_date', 'total_project_cost_1', 'status', 'team_leader', 'borrower_2',
    'disclosure_date', 'effective_date', 'implementing_agency', 'region', 
    'environmental_and_social_risk', 'last_stage_reached', 'last_update_date'
]

projects_df = projects_df[column_order]
print(f"Reordered to {len(column_order)} columns (dropped country)")
projects_df.head()

Reordered to 17 columns (dropped country)


,project_id,project_title,geography,short_summary,long_summary,closing_date,total_project_cost_1,status,team_leader,borrower_2,disclosure_date,effective_date,implementing_agency,region,environmental_and_social_risk,last_stage_reached,last_update_date
0,P119893,Ethiopia - Electricity Network Reinforcement a...,Ethiopia,The Electricity Network Reinforcement and Expa...,The Electricity Network Reinforcement and Expa...,"March 31, 2025",US$ 275.00 million,Closed,Abdulhakim Mohammed Abdisubhan,Federal Democratic Republic of Ethiopia,"December 22, 2011","January 4, 2013","Development Bank of Ethiopia,Ethiopia Electric...",Eastern and Southern Africa,Not Applicable,Bank Approved,"June 20, 2024"
1,P173506,"Congo, Democratic Republic of - Access Governa...",Democratic Republic of Congo,AGREE project expands renewable-based electric...,The AGREE project is a large water and energy ...,"September 30, 2029",US$ 939.00 million,Active,"Didier Makoso Tsasa , Fabrice Karl Bertholet, ...",DEMOCRATIC REPUBLIC OF CONGO,"January 6, 2021","May 18, 2023",Ministère des Ressources Hydrauliques et de l'...,Eastern and Southern Africa,High,Bank Approved,"June 17, 2024"
2,P176731,"Ethiopia - Power Sector Reform, Investment, an...",Ethiopia,This project supports Ethiopia's phased electr...,This project supports a phased electricity pro...,"August 30, 2030",US$ 537.00 million,Active,"Janina Franco , Abdulhakim Mohammed Abdisubhan...",Federal Democratic Republic of Ethiopia,"August 15, 2023","June 19, 2024","Ethiopia Electric Power,Ethiopia Electric Utility",Eastern and Southern Africa,High,Bank Approved,"June 16, 2023"
3,P507759,Mozambique - Accelerating Sustainable and Clea...,Mozambique,"ASCENT Mozambique finances on-grid expansion, ...",ASCENT Mozambique is Phase 10 of the ASCENT mu...,"December 31, 2030",US$ 0.00 million,Active,"Jenny Jing Chao , Maria Arango",Republic of Mozambique,"November 29, 2024","August 21, 2025","Electricidade de Moçambique (EDM),Fundo de Ene...",Eastern and Southern Africa,Substantial,Bank Approved,"March 22, 2025"
4,P180547,Eastern and Southern Africa - Accelerating Sus...,Eastern and Southern Africa (more than 20 Coun...,ASCENT accelerates sustainable and clean energ...,ASCENT is a regional program to accelerate sus...,"December 31, 2030",US$ 10000.00 million,Active,"Monali Ranade , Dana Rysankova, Alona Kazantseva",Common Market for Eastern and Southern Africa ...,"July 7, 2023","April 5, 2024",Common Market for Eastern and Southern Africa ...,Eastern and Southern Africa,Substantial,Bank Approved,"July 16, 2023"


### 1.06 Save project level data

In [16]:
# Create gold directory if it doesn't exist
GOLD_DIR.mkdir(parents=True, exist_ok=True)

# Save to CSV
output_file = GOLD_DIR / "project_level_data.csv"
projects_df.to_csv(output_file, index=False)

print(f"✓ Saved {len(projects_df)} projects to {output_file}")
print(f"  Columns: {len(projects_df.columns)}")
print(f"  File size: {output_file.stat().st_size / 1024:.1f} KB")

✓ Saved 98 projects to ../data/gold/project_level_data.csv
  Columns: 17
  File size: 245.0 KB


In [17]:
projects_df.head()

,project_id,project_title,geography,short_summary,long_summary,closing_date,total_project_cost_1,status,team_leader,borrower_2,disclosure_date,effective_date,implementing_agency,region,environmental_and_social_risk,last_stage_reached,last_update_date
0,P119893,Ethiopia - Electricity Network Reinforcement a...,Ethiopia,The Electricity Network Reinforcement and Expa...,The Electricity Network Reinforcement and Expa...,"March 31, 2025",US$ 275.00 million,Closed,Abdulhakim Mohammed Abdisubhan,Federal Democratic Republic of Ethiopia,"December 22, 2011","January 4, 2013","Development Bank of Ethiopia,Ethiopia Electric...",Eastern and Southern Africa,Not Applicable,Bank Approved,"June 20, 2024"
1,P173506,"Congo, Democratic Republic of - Access Governa...",Democratic Republic of Congo,AGREE project expands renewable-based electric...,The AGREE project is a large water and energy ...,"September 30, 2029",US$ 939.00 million,Active,"Didier Makoso Tsasa , Fabrice Karl Bertholet, ...",DEMOCRATIC REPUBLIC OF CONGO,"January 6, 2021","May 18, 2023",Ministère des Ressources Hydrauliques et de l'...,Eastern and Southern Africa,High,Bank Approved,"June 17, 2024"
2,P176731,"Ethiopia - Power Sector Reform, Investment, an...",Ethiopia,This project supports Ethiopia's phased electr...,This project supports a phased electricity pro...,"August 30, 2030",US$ 537.00 million,Active,"Janina Franco , Abdulhakim Mohammed Abdisubhan...",Federal Democratic Republic of Ethiopia,"August 15, 2023","June 19, 2024","Ethiopia Electric Power,Ethiopia Electric Utility",Eastern and Southern Africa,High,Bank Approved,"June 16, 2023"
3,P507759,Mozambique - Accelerating Sustainable and Clea...,Mozambique,"ASCENT Mozambique finances on-grid expansion, ...",ASCENT Mozambique is Phase 10 of the ASCENT mu...,"December 31, 2030",US$ 0.00 million,Active,"Jenny Jing Chao , Maria Arango",Republic of Mozambique,"November 29, 2024","August 21, 2025","Electricidade de Moçambique (EDM),Fundo de Ene...",Eastern and Southern Africa,Substantial,Bank Approved,"March 22, 2025"
4,P180547,Eastern and Southern Africa - Accelerating Sus...,Eastern and Southern Africa (more than 20 Coun...,ASCENT accelerates sustainable and clean energ...,ASCENT is a regional program to accelerate sus...,"December 31, 2030",US$ 10000.00 million,Active,"Monali Ranade , Dana Rysankova, Alona Kazantseva",Common Market for Eastern and Southern Africa ...,"July 7, 2023","April 5, 2024",Common Market for Eastern and Southern Africa ...,Eastern and Southern Africa,Substantial,Bank Approved,"July 16, 2023"


## 2. Project-Occupation Level Data

### 2.01 Load and combine unique ESCO-NACE-O*NET data

In [18]:
# Directory containing unique ESCO-NACE-O*NET files
esco_nace_onet_dir = SILVER_DIR / "unique_esco_nace_onet_csv"

# Load all CSV files for projects in projects_df
all_data = []

for csv_file in esco_nace_onet_dir.glob("*_esco_nace_onet.csv"):
    # Extract project_id from filename (e.g., P075941_esco_nace_onet.csv -> P075941)
    project_id = csv_file.stem.replace("_esco_nace_onet", "")
    
    # Only load if project is in projects_df
    if project_id in projects_df['project_id'].values:
        df = pd.read_csv(csv_file)
        df['project_id'] = project_id
        all_data.append(df)
        print(f"Loaded {project_id}: {len(df)} rows")

# Combine all dataframes
if all_data:
    combined_df = pd.concat(all_data, ignore_index=True)
    print(f"\n✓ Combined {len(all_data)} project files")
    print(f"Total rows: {len(combined_df)}")
    print(f"Columns: {list(combined_df.columns)}")
else:
    print("No data files found")
    combined_df = pd.DataFrame()

combined_df.head()

Loaded P166796: 61 rows
Loaded P179631: 73 rows
Loaded P164885: 38 rows
Loaded P157096: 44 rows
Loaded P166936: 61 rows
Loaded P179267: 58 rows
Loaded P176698: 51 rows
Loaded P160708: 61 rows
Loaded P178161: 48 rows
Loaded P511453: 45 rows
Loaded P506061: 30 rows
Loaded P163752: 46 rows
Loaded P173749: 51 rows
Loaded P176620: 37 rows
Loaded P128768: 52 rows
Loaded P173416: 50 rows
Loaded P162933: 43 rows
Loaded P176776: 63 rows
Loaded P160395: 48 rows
Loaded P181221: 47 rows
Loaded P162245: 55 rows
Loaded P181160: 43 rows
Loaded P164044: 35 rows
Loaded P505173: 62 rows
Loaded P152755: 41 rows
Loaded P503941: 38 rows
Loaded P161015: 50 rows
Loaded P173088: 62 rows
Loaded P160427: 54 rows
Loaded P166785: 61 rows
Loaded P133312: 48 rows
Loaded P165704: 35 rows
Loaded P170236: 40 rows
Loaded P164001: 44 rows
Loaded P173506: 46 rows
Loaded P171742: 77 rows
Loaded P157055: 42 rows
Loaded P162760: 49 rows
Loaded P163881: 49 rows
Loaded P169332: 67 rows
Loaded P172594: 44 rows
Loaded P156208: 

,esco_id,esco_label,esco_description,group_code,group_label_en,division_code,division_label_en,section_code,section_label_en,pad_occupations,pad_activities,pad_skills,pad_quotes,onet_job_zone,onet_job_zone_label,onet_job_zone_est,project_id
0,04f39bfa-bc03-4480-98bc-b18ce4fe4b4b,accounting manager,Accounting managers assume responsibility for ...,692.0,"69.2 Accounting, bookkeeping and auditing acti...",69.0,69 Legal and accounting activities,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...","""accounts payable specialist""","""Process and schedule payments to suppliers to...","""supplier payment processing"", ""cash flow plan...",ANNEX 5: FINANCIAL SITUATION OF THE POWER SECT...,4.0,4: Considerable Preparation Needed,llm,P166796
1,0561328b-875b-4ae2-9ba1-9af9049aef01,procurement category specialist,Procurement category specialists are experts i...,702.0,70.2 Business and other management consultancy...,70.0,70 Activities of head offices and management c...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...","""procurement specialist""","""Revise and implement efficient fuel procureme...","""competitive tendering"", ""supplier evaluation""...",ANNEX 5: FINANCIAL SITUATION OF THE POWER SECT...,2.0,2: Some Preparation Needed,minimum,P166796
2,056bef79-c125-47ab-b6b9-8eed05c9458c,telecommunications technician,"Telecommunications technicians install, test, ...",951.0,95.1 Repair and maintenance of computers and c...,95.0,"95 Repair and maintenance of computers, person...",T,T OTHER SERVICE ACTIVITIES,"""call center technician""","""Install the customer call center as part of c...","""telephony system installation"", ""call center ...","VII. RESULTS FRAMEWORK AND MONITORING: ""The ca...",2.0,2: Some Preparation Needed,minimum,P166796
3,08f5481e-9233-4ef2-9e4b-945224879ce5,commissioning technician,Commissioning technicians work with commission...,712.0,71.2 Technical testing and analysis,71.0,71 Architectural and engineering activities; t...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...","""substation technician""","""Perform hands-on rehabilitation, installation...","""switchgear installation"", ""protection relay t...","ANNEX 4: ECONOMIC AND FINANCIAL ANALYSIS: ""The...",3.0,3: Medium Preparation Needed,llm,P166796
4,0ba06640-e0ac-4911-9e43-289a8e41651e,corporate trainer,"Corporate trainers train, coach, and guide emp...",855.0,85.5 Other education,85.0,85 Education,Q,Q EDUCATION,"""training specialist""","""Deliver capacity strengthening activities to ...","""training needs assessment"", ""curriculum devel...","ANNEX 4: ECONOMIC AND FINANCIAL ANALYSIS: ""The...",4.0,4: Considerable Preparation Needed,minimum,P166796


### 2.02 Rename and reorder columns

In [19]:
# Rename columns to use more descriptive names
combined_df = combined_df.rename(columns={
    'esco_label': 'occupation_esco',
    'division_code': 'industry_division_code',
    'division_label_en': 'industry_division_label',
    'section_code': 'industry_cat_code',
    'section_label_en': 'industry_cat_label',
    'pad_occupations': 'pad_job_titles'
})

print("Renamed columns")

# Select and reorder columns
columns_to_keep = [
    'project_id', 'esco_id', 'occupation_esco', 'esco_description', 
    'industry_cat_code', 'industry_cat_label', 
    'industry_division_code', 'industry_division_label', 
    'onet_job_zone', 'onet_job_zone_label', 'onet_job_zone_est', 
    'pad_job_titles', 'pad_activities', 'pad_skills', 'pad_quotes'
]

combined_df = combined_df[columns_to_keep]


Renamed columns


In [20]:

print(f"Selected and reordered to {len(columns_to_keep)} columns")
print(f"Final shape: {combined_df.shape}")
combined_df.head(100)

Selected and reordered to 15 columns
Final shape: (4933, 15)


,project_id,esco_id,occupation_esco,esco_description,industry_cat_code,industry_cat_label,industry_division_code,industry_division_label,onet_job_zone,onet_job_zone_label,onet_job_zone_est,pad_job_titles,pad_activities,pad_skills,pad_quotes
0,P166796,04f39bfa-bc03-4480-98bc-b18ce4fe4b4b,accounting manager,Accounting managers assume responsibility for ...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...",69.0,69 Legal and accounting activities,4.0,4: Considerable Preparation Needed,llm,"""accounts payable specialist""","""Process and schedule payments to suppliers to...","""supplier payment processing"", ""cash flow plan...",ANNEX 5: FINANCIAL SITUATION OF THE POWER SECT...
1,P166796,0561328b-875b-4ae2-9ba1-9af9049aef01,procurement category specialist,Procurement category specialists are experts i...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...",70.0,70 Activities of head offices and management c...,2.0,2: Some Preparation Needed,minimum,"""procurement specialist""","""Revise and implement efficient fuel procureme...","""competitive tendering"", ""supplier evaluation""...",ANNEX 5: FINANCIAL SITUATION OF THE POWER SECT...
2,P166796,056bef79-c125-47ab-b6b9-8eed05c9458c,telecommunications technician,"Telecommunications technicians install, test, ...",T,T OTHER SERVICE ACTIVITIES,95.0,"95 Repair and maintenance of computers, person...",2.0,2: Some Preparation Needed,minimum,"""call center technician""","""Install the customer call center as part of c...","""telephony system installation"", ""call center ...","VII. RESULTS FRAMEWORK AND MONITORING: ""The ca..."
3,P166796,08f5481e-9233-4ef2-9e4b-945224879ce5,commissioning technician,Commissioning technicians work with commission...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...",71.0,71 Architectural and engineering activities; t...,3.0,3: Medium Preparation Needed,llm,"""substation technician""","""Perform hands-on rehabilitation, installation...","""switchgear installation"", ""protection relay t...","ANNEX 4: ECONOMIC AND FINANCIAL ANALYSIS: ""The..."
4,P166796,0ba06640-e0ac-4911-9e43-289a8e41651e,corporate trainer,"Corporate trainers train, coach, and guide emp...",Q,Q EDUCATION,85.0,85 Education,4.0,4: Considerable Preparation Needed,minimum,"""training specialist""","""Deliver capacity strengthening activities to ...","""training needs assessment"", ""curriculum devel...","ANNEX 4: ECONOMIC AND FINANCIAL ANALYSIS: ""The..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,P179631,8213b4bc-60ce-4ffc-9e00-9758fb2003c6,monitoring and evaluation officer,Monitoring and evaluation officers are respons...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...",70.0,70 Activities of head offices and management c...,4.0,4: Considerable Preparation Needed,llm,"""capacity building specialist"", ""impact evalua...","""Support adoption and operation of digital pla...","""digital dashboard development"", ""data managem...","I. STRATEGIC CONTEXT: ""ASCENT will launch comp..."
96,P179631,836f0a96-7546-4003-9fc8-714fc79199d4,contract engineer,Contract engineers combine technical knowledge...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...",71.0,71 Architectural and engineering activities; t...,4.0,4: Considerable Preparation Needed,llm,"""owners engineer""","""Provide detailed designs, procurement support...","""detailed engineering design"", ""contract manag...","ANNEX 5: Somalia (P181341): ""An owner's engine..."
97,P179631,86ca306c-ab99-420a-9e2a-aa73c5c4de22,electrical engineer,Electrical engineers design and develop electr...,C,C MANUFACTURING,27.0,27 Manufacture of electrical equipment,4.0,4: Considerable Preparation Needed,unique,"""electrical engineer""","""Provide technical assistance and engineering ...","""power systems engineering"", ""grid integration...","III. PROJECT DESCRIPTION: ""Pillar 2: Expanding..."
98,P179631,89330b57-6a30-40e1-b623-50f47a4b0c34,grants management officer,Grants management officers work professionally...,P,P PUBLIC AD

### 2.03 Save occupation level data

In [21]:
# Save to CSV
output_file = GOLD_DIR / "occupation_level_data.csv"
combined_df.to_csv(output_file, index=False)

print(f"✓ Saved {len(combined_df)} occupation records to {output_file}")
print(f"  Columns: {len(combined_df.columns)}")
print(f"  File size: {output_file.stat().st_size / 1024:.1f} KB")

✓ Saved 4933 occupation records to ../data/gold/occupation_level_data.csv
  Columns: 15
  File size: 8729.7 KB


## 3. Project-Occupation-Skills Level Data

### 3.01 Load and combine ESCO-NACE with skills data

In [22]:
# Directory containing ESCO-NACE with skills files
esco_nace_skills_dir = SILVER_DIR / "esco_nace_w_skills_csv"

# Load all CSV files for projects in projects_df
all_skills_data = []

for csv_file in esco_nace_skills_dir.glob("*_esco_nace_with_skills.csv"):
    # Extract project_id from filename (e.g., P075941_esco_nace_with_skills.csv -> P075941)
    project_id = csv_file.stem.replace("_esco_nace_with_skills", "")
    
    # Only load if project is in projects_df
    if project_id in projects_df['project_id'].values:
        df = pd.read_csv(csv_file)
        df['project_id'] = project_id
        all_skills_data.append(df)
        print(f"Loaded {project_id}: {len(df)} rows")

# Combine all dataframes
if all_skills_data:
    skills_df = pd.concat(all_skills_data, ignore_index=True)
    print(f"\n✓ Combined {len(all_skills_data)} project files")
    print(f"Total rows: {len(skills_df)}")
    print(f"Columns: {list(skills_df.columns)}")
else:
    print("No data files found")
    skills_df = pd.DataFrame()

skills_df.head()

Loaded P177099: 1890 rows
Loaded P166936: 1393 rows
Loaded P164885: 1088 rows
Loaded P507759: 827 rows
Loaded P171742: 1746 rows
Loaded P501343: 1060 rows
Loaded P176731: 1096 rows
Loaded P163881: 1150 rows
Loaded P168185: 1245 rows
Loaded P178161: 1279 rows
Loaded P174034: 1784 rows
Loaded P176683: 1294 rows
Loaded P504762: 1440 rows
Loaded P075941: 1429 rows
Loaded P149683: 1108 rows
Loaded P161015: 1227 rows
Loaded P146830: 866 rows
Loaded P180547: 1515 rows
Loaded P160377: 900 rows
Loaded P160009: 1561 rows
Loaded P173088: 1580 rows
Loaded P157055: 972 rows
Loaded P166796: 1430 rows
Loaded P172594: 1012 rows
Loaded P169332: 1541 rows
Loaded P166805: 1165 rows
Loaded P510382: 732 rows
Loaded P170236: 939 rows
Loaded P181328: 1716 rows
Loaded P162245: 1344 rows
Loaded P181494: 979 rows
Loaded P161885: 1515 rows
Loaded P173416: 1292 rows
Loaded P144135: 896 rows
Loaded P506269: 964 rows
Loaded P173749: 1302 rows
Loaded P179380: 1104 rows
Loaded P160708: 1428 rows
Loaded P173506: 1053 

,esco_id,esco_label,esco_description,group_code,group_label_en,division_code,division_label_en,section_code,section_label_en,pad_occupations,...,relationType,skillType,skillUri,skillLabel,occupation_uuid,skill_code,esco_num,relevant,top_five,project_id
0,0561328b-875b-4ae2-9ba1-9af9049aef01,procurement category specialist,Procurement category specialists are experts i...,702.0,70.2 Business and other management consultancy...,70.0,70 Activities of head offices and management c...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...","""procurement specialist"", ""senior procurement ...",...,essential,skill/competence,http://data.europa.eu/esco/skill/23ac233d-84ad...,monitor developments in field of expertise,0561328b-875b-4ae2-9ba1-9af9049aef01,23ac233d-84ad-4517-b0f5-8ca19ba2614e,1,True,False,P177099
1,0561328b-875b-4ae2-9ba1-9af9049aef01,procurement category specialist,Procurement category specialists are experts i...,702.0,70.2 Business and other management consultancy...,70.0,70 Activities of head offices and management c...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...","""procurement specialist"", ""senior procurement ...",...,essential,skill/competence,http://data.europa.eu/esco/skill/305dc408-68a6...,develop performance orientation in public admi...,0561328b-875b-4ae2-9ba1-9af9049aef01,305dc408-68a6-4f63-9ee6-d0b26c8abad0,1,True,False,P177099
2,0561328b-875b-4ae2-9ba1-9af9049aef01,procurement category specialist,Procurement category specialists are experts i...,702.0,70.2 Business and other management consultancy...,70.0,70 Activities of head offices and management c...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...","""procurement specialist"", ""senior procurement ...",...,essential,skill/competence,http://data.europa.eu/esco/skill/31599ac3-e03c...,draft procurement technical specifications,0561328b-875b-4ae2-9ba1-9af9049aef01,31599ac3-e03c-42f8-8f08-b549af8234f2,1,True,True,P177099
3,0561328b-875b-4ae2-9ba1-9af9049aef01,procurement category specialist,Procurement category specialists are experts i...,702.0,70.2 Business and other management consultancy...,70.0,70 Activities of head offices and management c...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...","""procurement specialist"", ""senior procurement ...",...,essential,knowledge,http://data.europa.eu/esco/skill/3bc08c26-2f98...,procurement lifecycle,0561328b-875b-4ae2-9ba1-9af9049aef01,3bc08c26-2f98-4730-bbb6-ec4a8566e3cf,1,True,True,P177099
4,0561328b-875b-4ae2-9ba1-9af9049aef01,procurement category specialist,Procurement category specialists are experts i...,702.0,70.2 Business and other management consultancy...,70.0,70 Activities of head offices and management c...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...","""procurement specialist"", ""senior procurement ...",...,essential,skill/competence,http://data.europa.eu/esco/skill/3cd35f5d-ce6d...,maintain relationship with suppliers,0561328b-875b-4ae2-9ba1-9af9049aef01,3cd35f5d-ce6d-4f14-9a09-53d7a28d834c,1,True,False,P177099


### 3.02 Filter relevant skills

In [23]:
# Keep only skills where relevant=True
print(f"Total rows before filtering: {len(skills_df)}")
skills_df = skills_df[skills_df['relevant'] == True]
print(f"Rows after filtering for relevant=True: {len(skills_df)}")
skills_df.head()

Total rows before filtering: 117528
Rows after filtering for relevant=True: 99274


,esco_id,esco_label,esco_description,group_code,group_label_en,division_code,division_label_en,section_code,section_label_en,pad_occupations,...,relationType,skillType,skillUri,skillLabel,occupation_uuid,skill_code,esco_num,relevant,top_five,project_id
0,0561328b-875b-4ae2-9ba1-9af9049aef01,procurement category specialist,Procurement category specialists are experts i...,702.0,70.2 Business and other management consultancy...,70.0,70 Activities of head offices and management c...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...","""procurement specialist"", ""senior procurement ...",...,essential,skill/competence,http://data.europa.eu/esco/skill/23ac233d-84ad...,monitor developments in field of expertise,0561328b-875b-4ae2-9ba1-9af9049aef01,23ac233d-84ad-4517-b0f5-8ca19ba2614e,1,True,False,P177099
1,0561328b-875b-4ae2-9ba1-9af9049aef01,procurement category specialist,Procurement category specialists are experts i...,702.0,70.2 Business and other management consultancy...,70.0,70 Activities of head offices and management c...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...","""procurement specialist"", ""senior procurement ...",...,essential,skill/competence,http://data.europa.eu/esco/skill/305dc408-68a6...,develop performance orientation in public admi...,0561328b-875b-4ae2-9ba1-9af9049aef01,305dc408-68a6-4f63-9ee6-d0b26c8abad0,1,True,False,P177099
2,0561328b-875b-4ae2-9ba1-9af9049aef01,procurement category specialist,Procurement category specialists are experts i...,702.0,70.2 Business and other management consultancy...,70.0,70 Activities of head offices and management c...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...","""procurement specialist"", ""senior procurement ...",...,essential,skill/competence,http://data.europa.eu/esco/skill/31599ac3-e03c...,draft procurement technical specifications,0561328b-875b-4ae2-9ba1-9af9049aef01,31599ac3-e03c-42f8-8f08-b549af8234f2,1,True,True,P177099
3,0561328b-875b-4ae2-9ba1-9af9049aef01,procurement category specialist,Procurement category specialists are experts i...,702.0,70.2 Business and other management consultancy...,70.0,70 Activities of head offices and management c...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...","""procurement specialist"", ""senior procurement ...",...,essential,knowledge,http://data.europa.eu/esco/skill/3bc08c26-2f98...,procurement lifecycle,0561328b-875b-4ae2-9ba1-9af9049aef01,3bc08c26-2f98-4730-bbb6-ec4a8566e3cf,1,True,True,P177099
4,0561328b-875b-4ae2-9ba1-9af9049aef01,procurement category specialist,Procurement category specialists are experts i...,702.0,70.2 Business and other management consultancy...,70.0,70 Activities of head offices and management c...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...","""procurement specialist"", ""senior procurement ...",...,essential,skill/competence,http://data.europa.eu/esco/skill/3cd35f5d-ce6d...,maintain relationship with suppliers,0561328b-875b-4ae2-9ba1-9af9049aef01,3cd35f5d-ce6d-4f14-9a09-53d7a28d834c,1,True,False,P177099


### 3.03 Process broader skill relations

In [24]:
# Load broader skill relations
skill_relations_file = DATA_DIR / "bronze" / "esco" / "broaderRelationsSkillPillar_en.csv"
skill_relations = pd.read_csv(skill_relations_file)

print(f"Loaded skill relations: {skill_relations.shape}")
print(f"Columns: {list(skill_relations.columns)}")
skill_relations.head()

Loaded skill relations: (20819, 6)
Columns: ['conceptType', 'conceptUri', 'conceptLabel', 'broaderType', 'broaderUri', 'broaderLabel']


,conceptType,conceptUri,conceptLabel,broaderType,broaderUri,broaderLabel
0,SkillGroup,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications,SkillGroup,http://data.europa.eu/esco/skill/c46fcb45-5c14...,knowledge
1,SkillGroup,http://data.europa.eu/esco/isced-f/000,generic programmes and qualifications not furt...,SkillGroup,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications
2,SkillGroup,http://data.europa.eu/esco/isced-f/0000,generic programmes and qualifications not furt...,SkillGroup,http://data.europa.eu/esco/isced-f/000,generic programmes and qualifications not furt...
3,SkillGroup,http://data.europa.eu/esco/isced-f/001,basic programmes and qualifications,SkillGroup,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications
4,SkillGroup,http://data.europa.eu/esco/isced-f/0011,basic programmes and qualifications,SkillGroup,http://data.europa.eu/esco/isced-f/001,basic programmes and qualifications


In [25]:
# Get level 0 and 1 skill categories from broader relations
import numpy as np
from functools import lru_cache

# Reload skill_relations to ensure we have the original columns
skill_relations = pd.read_csv(DATA_DIR / "bronze" / "esco" / "broaderRelationsSkillPillar_en.csv")

# First rename columns in skill_relations
skill_relations_clean = skill_relations[['conceptUri', 'conceptLabel', 'broaderUri', 'broaderLabel']].rename(
    columns={'conceptUri': 'skill_uri',
             'conceptLabel': 'skill_label',
             'broaderUri': 'skills_cat_uri', 
             'broaderLabel': 'broader_label'}
)

print(f"Renamed skill_relations columns: {list(skill_relations_clean.columns)}")

# Build parents map: concept -> list of parents (skills_cat_uri)
parents_map = (
    skill_relations_clean.groupby("skill_uri", sort=False)["skills_cat_uri"]
      .apply(lambda s: list(dict.fromkeys(s.tolist())))
      .to_dict()
)

# Build label map for all URIs (concepts and their parent categories)
label_map = {}

# Map concept URIs to their labels
label_map.update(
    skill_relations_clean.drop_duplicates("skill_uri")
    .set_index("skill_uri")["skill_label"]
    .to_dict()
)

# Map broader URIs (parent category URIs) to their labels
for _, r in skill_relations_clean[["skills_cat_uri", "broader_label"]].drop_duplicates().iterrows():
    label_map.setdefault(r["skills_cat_uri"], r["broader_label"])

# Debug: Check the specific skill mentioned
test_uri = "http://data.europa.eu/esco/skill/c8b5b67b-d8d9-4857-bb26-1e35faf2ce22"
if test_uri in parents_map:
    print(f"\nDirect parents of 'clean road vehicles' in data:")
    for parent in parents_map[test_uri]:
        print(f"  - {label_map.get(parent, parent)}")

@lru_cache(None)
def get_paths(uri: str, cap: int = 500):
    """
    All paths (root -> ... -> uri), inclusive.
    A root is any uri that does not appear as a skill_uri with its own parent info.
    """
    if uri not in parents_map:
        return [(uri,)]
    out = []
    for parent in parents_map[uri]:
        for base in get_paths(parent):
            out.append(base + (uri,))
            if len(out) >= cap:
                return out
    return out

# Build rows with level 0 and level 1 information
rows = []
for concept_uri in parents_map.keys():
    for path in get_paths(concept_uri):
        level0 = path[0]
        level1 = path[1] if len(path) > 1 else np.nan
        rows.append({
            "skill_uri": concept_uri,
            "skill_label": label_map.get(concept_uri),
            "level_0_uri": level0,
            "level_0_label": label_map.get(level0),
            "level_1_uri": level1,
            "level_1_label": label_map.get(level1) if isinstance(level1, str) else np.nan,
        })

skill_levels = pd.DataFrame(rows).drop_duplicates()

print(f"\nCreated skill levels dataframe: {skill_levels.shape}")
print(f"Unique concepts: {skill_levels['skill_uri'].nunique()}")

# Debug: Show all paths for the test skill
if test_uri in parents_map:
    test_paths = skill_levels[skill_levels['skill_uri'] == test_uri][['level_0_label', 'level_1_label']]
    print(f"\nPaths found for 'clean road vehicles':")
    for idx, row in test_paths.iterrows():
        print(f"  {row['level_0_label']} > {row['level_1_label']}")

print("\nPaths per concept distribution:")
print(skill_levels.groupby("skill_uri").size().value_counts().head(10))
skill_levels.head(20)

Renamed skill_relations columns: ['skill_uri', 'skill_label', 'skills_cat_uri', 'broader_label']

Direct parents of 'clean road vehicles' in data:
  - cleaning tools, equipment, workpieces and vehicles
  - clean vehicle interiors
  - follow safety precautions in work practices

Created skill levels dataframe: (19772, 6)
Unique concepts: 14575

Paths found for 'clean road vehicles':
  skills > handling and moving
  skills > working with machinery and specialised equipment
  skills > constructing
  skills > management skills
  transversal skills and competences > social and communication skills and competences
  skills > assisting and caring
  transversal skills and competences > life skills and competences

Paths per concept distribution:
1    10761
2     2846
3      683
4      192
5       59
6       31
7        3
Name: count, dtype: int64


,skill_uri,skill_label,level_0_uri,level_0_label,level_1_uri,level_1_label
0,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications,http://data.europa.eu/esco/skill/c46fcb45-5c14...,knowledge,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications
1,http://data.europa.eu/esco/isced-f/000,generic programmes and qualifications not furt...,http://data.europa.eu/esco/skill/c46fcb45-5c14...,knowledge,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications
2,http://data.europa.eu/esco/isced-f/0000,generic programmes and qualifications not furt...,http://data.europa.eu/esco/skill/c46fcb45-5c14...,knowledge,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications
3,http://data.europa.eu/esco/isced-f/001,basic programmes and qualifications,http://data.europa.eu/esco/skill/c46fcb45-5c14...,knowledge,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications
4,http://data.europa.eu/esco/isced-f/0011,basic programmes and qualifications,http://data.europa.eu/esco/skill/c46fcb45-5c14...,knowledge,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications
5,http://data.europa.eu/esco/isced-f/002,literacy and numeracy,http://data.europa.eu/esco/skill/c46fcb45-5c14...,knowledge,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications
6,http://data.europa.eu/esco/isced-f/0021,literacy and numeracy,http://data.europa.eu/esco/skill/c46fcb45-5c14...,knowledge,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications
7,http://data.europa.eu/esco/isced-f/003,personal skills and development,http://data.europa.eu/esco/skill/c46fcb45-5c14...,knowledge,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications
8,http://data.europa.eu/esco/isced-f/0031,personal skills and development,http://data.europa.eu/esco/skill/c46fcb45-5c14...,knowledge,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications
9,http://data.europa.eu/esco/isced-f/009,generic programmes and qualifications not else...,http://data.europa.eu/esco/skill/c46fcb45-5c14...,knowledge,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications


In [26]:
# Find skill URIs that appear multiple times (have multiple parent paths)
uri_counts = skill_levels.groupby('skill_uri').size()
duplicate_uris = uri_counts[uri_counts > 1].index

print(f"Found {len(duplicate_uris)} skill URIs with multiple parent paths")
print(f"Total duplicate rows: {uri_counts[uri_counts > 1].sum()}")

# Show the rows for these duplicated URIs
duplicated_skills = skill_levels[skill_levels['skill_uri'].isin(duplicate_uris)].sort_values('skill_uri')

print(f"\nShowing {len(duplicated_skills)} rows with duplicate skill URIs:")
duplicated_skills

Found 3814 skill URIs with multiple parent paths
Total duplicate rows: 9011

Showing 9011 rows with duplicate skill URIs:


,skill_uri,skill_label,level_0_uri,level_0_label,level_1_uri,level_1_label
636,http://data.europa.eu/esco/skill/0005c151-5b5a...,manage musical staff,http://data.europa.eu/esco/skill/335228d2-297d...,skills,http://data.europa.eu/esco/skill/869fc2ce-478f...,management skills
637,http://data.europa.eu/esco/skill/0005c151-5b5a...,manage musical staff,http://data.europa.eu/esco/skill/04a13491-b58c...,transversal skills and competences,http://data.europa.eu/esco/skill/552c4f35-a2d1...,social and communication skills and competences
644,http://data.europa.eu/esco/skill/0007bdc2-dd15...,control compliance of railway vehicles regulat...,http://data.europa.eu/esco/skill/335228d2-297d...,skills,http://data.europa.eu/esco/skill/c73521be-c039...,assisting and caring
645,http://data.europa.eu/esco/skill/0007bdc2-dd15...,control compliance of railway vehicles regulat...,http://data.europa.eu/esco/skill/335228d2-297d...,skills,http://data.europa.eu/esco/skill/0a2d70ee-d435...,information skills
654,http://data.europa.eu/esco/skill/00298d97-3dc3...,lead police investigations,http://data.europa.eu/esco/skill/335228d2-297d...,skills,http://data.europa.eu/esco/skill/0a2d70ee-d435...,information skills
...,...,...,...,...,...,...
24738,http://data.europa.eu/esco/skill/ff7f57d7-084b...,write Occitan,http://data.europa.eu/esco/skill/e35a5936-091d...,language skills and knowledge,http://data.europa.eu/esco/skill/43f425aa-f45d...,languages
24140,http://data.europa.eu/esco/skill/ff862044-d49e...,analyse work-related written reports,http://data.europa.eu/esco/skill/335228d2-297d...,skills,http://data.europa.eu/esco/skill/0a2d70ee-d435...,information skills
24141,http://data.europa.eu/esco/skill/ff862044-d49e...,analyse work-related written reports,http://data.europa.eu/esco/skill/04a13491-b58c...,transversal skills and competences,http://data.europa.eu/esco/skill/8267ecb5-c976...,thinking skills and competences
24171,http://data.europa.eu/esco/skill/fff5bc45-b506...,coordinate construction activities,http://data.europa.eu/esco/skill/04a13491-b58c...,transversal skills and competences,http://data.europa.eu/esco/skill/552c4f35-a2d1...,social and communication skills and competences


In [27]:
# Identify duplicated skill URIs that have both transversal and non-transversal rows
duplicated_uris = skill_levels['skill_uri'].duplicated(keep=False)
transversal_mask = skill_levels['level_0_label'] == 'transversal skills and competences'

# For each skill_uri, check if it has both transversal and non-transversal rows
skill_uri_groups = skill_levels.groupby('skill_uri')['level_0_label'].apply(
    lambda x: ('transversal skills and competences' in x.values) and (len(x.unique()) > 1)
)
uris_with_both = skill_uri_groups[skill_uri_groups].index

print(f"Skill URIs with both transversal and non-transversal rows: {len(uris_with_both)}")
print(f"Current total rows: {len(skill_levels)}")
print(f"Current duplicate skill URIs: {skill_levels['skill_uri'].duplicated(keep=False).sum()}")

# Drop transversal rows only for URIs that have other non-transversal rows
rows_to_drop = skill_levels['skill_uri'].isin(uris_with_both) & transversal_mask
print(f"\nRows to drop: {rows_to_drop.sum()}")

# Create filtered dataframe
skill_levels_filtered = skill_levels[~rows_to_drop].copy()

print(f"\nAfter filtering:")
print(f"Total rows: {len(skill_levels_filtered)}")
print(f"Duplicate skill URIs: {skill_levels_filtered['skill_uri'].duplicated(keep=False).sum()}")

# Show remaining duplicates
remaining_duplicates = skill_levels_filtered.groupby('skill_uri').size()
remaining_duplicates = remaining_duplicates[remaining_duplicates > 1]
print(f"\nRemaining skill URIs with multiple rows: {len(remaining_duplicates)}")

skill_levels_filtered.head()

Skill URIs with both transversal and non-transversal rows: 2895
Current total rows: 19772
Current duplicate skill URIs: 9011

Rows to drop: 3165

After filtering:
Total rows: 16607
Duplicate skill URIs: 3733

Remaining skill URIs with multiple rows: 1701


,skill_uri,skill_label,level_0_uri,level_0_label,level_1_uri,level_1_label
0,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications,http://data.europa.eu/esco/skill/c46fcb45-5c14...,knowledge,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications
1,http://data.europa.eu/esco/isced-f/000,generic programmes and qualifications not furt...,http://data.europa.eu/esco/skill/c46fcb45-5c14...,knowledge,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications
2,http://data.europa.eu/esco/isced-f/0000,generic programmes and qualifications not furt...,http://data.europa.eu/esco/skill/c46fcb45-5c14...,knowledge,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications
3,http://data.europa.eu/esco/isced-f/001,basic programmes and qualifications,http://data.europa.eu/esco/skill/c46fcb45-5c14...,knowledge,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications
4,http://data.europa.eu/esco/isced-f/0011,basic programmes and qualifications,http://data.europa.eu/esco/skill/c46fcb45-5c14...,knowledge,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications


In [28]:
# Drop level 0 columns
skill_levels_wide = skill_levels_filtered.drop(columns=['level_0_uri', 'level_0_label'])

# Drop duplicates
skill_levels_wide = skill_levels_wide.drop_duplicates()

print(f"After dropping level 0 columns and duplicates: {skill_levels_wide.shape}")

# Sort by level_1_uri
skill_levels_wide = skill_levels_wide.sort_values('level_1_uri')

# Group by skill_uri and skill_label, then enumerate level_1 categories
grouped = skill_levels_wide.groupby(['skill_uri', 'skill_label'], sort=False)

# Create wide format with numbered columns
rows = []
for (skill_uri, skill_label), group in grouped:
    row = {'skill_uri': skill_uri, 'skill_label': skill_label}
    
    # Add numbered level_1_uri and level_1_label columns
    for idx, (_, r) in enumerate(group.iterrows(), start=1):
        row[f'level_1_uri_{idx}'] = r['level_1_uri']
        row[f'level_1_label_{idx}'] = r['level_1_label']
    
    rows.append(row)

skill_levels_wide = pd.DataFrame(rows)

print(f"\nReshaped to wide format: {skill_levels_wide.shape}")
print(f"Unique skills: {skill_levels_wide['skill_uri'].nunique()}")
print(f"\nColumn names: {list(skill_levels_wide.columns)}")
skill_levels_wide.head(20)

After dropping level 0 columns and duplicates: (16607, 4)

Reshaped to wide format: (14575, 12)
Unique skills: 14575

Column names: ['skill_uri', 'skill_label', 'level_1_uri_1', 'level_1_label_1', 'level_1_uri_2', 'level_1_label_2', 'level_1_uri_3', 'level_1_label_3', 'level_1_uri_4', 'level_1_label_4', 'level_1_uri_5', 'level_1_label_5']


,skill_uri,skill_label,level_1_uri_1,level_1_label_1,level_1_uri_2,level_1_label_2,level_1_uri_3,level_1_label_3,level_1_uri_4,level_1_label_4,level_1_uri_5,level_1_label_5
0,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,http://data.europa.eu/esco/skill/b17f4305-741a...,business communication,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,http://data.europa.eu/esco/skill/d5145a9a-602e...,leadership principles,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,http://data.europa.eu/esco/skill/15d76317-c71a...,communication,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,http://data.europa.eu/esco/skill/a6e1e0a0-07f8...,hand gestures,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,http://data.europa.eu/esco/skill/a5b0cd5c-e13a...,teamwork principles,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,http://data.europa.eu/esco/skill/cf6ca1fc-f2be...,leadership in nursing,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications,http://data.europa.eu/esco/isced-f/09,health and welfare,NaN,NaN,NaN,NaN,NaN,NaN
7,http://data.europa.eu/esco/skill/519e801b-3cc4...,personal development,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,http://data.europa.eu/esco/skill/a0cad388-3c4c...,assertiveness,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,http://data.europa.eu/esco/skill/fe30a4b0-1a99...,communication principles,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
# Show rows where level_1_uri_2 is not missing
print("No duplicates: " + str(len(skill_levels_wide[skill_levels_wide['level_1_uri_2'].isna()])))
print("Has duplicates: " + str(len(skill_levels_wide[skill_levels_wide['level_1_uri_2'].notna()])))

No duplicates: 12874
Has duplicates: 1701


In [30]:
# Drop level_1 columns 2-5
columns_to_drop = [
    'level_1_uri_2', 'level_1_label_2',
    'level_1_uri_3', 'level_1_label_3',
    'level_1_uri_4', 'level_1_label_4',
    'level_1_uri_5', 'level_1_label_5'
]
skill_levels_wide = skill_levels_wide.drop(columns=columns_to_drop)

# Rename level_1_uri_1 and level_1_label_1
skill_levels_wide = skill_levels_wide.rename(columns={
    'level_1_uri_1': 'skill_category_uri',
    'level_1_label_1': 'skill_category_label'
})

print(f"Dropped {len(columns_to_drop)} columns and renamed level_1 columns")
print(f"Final shape: {skill_levels_wide.shape}")
print(f"Columns: {list(skill_levels_wide.columns)}")
skill_levels_wide.head()

Dropped 8 columns and renamed level_1 columns
Final shape: (14575, 4)
Columns: ['skill_uri', 'skill_label', 'skill_category_uri', 'skill_category_label']


,skill_uri,skill_label,skill_category_uri,skill_category_label
0,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications
1,http://data.europa.eu/esco/skill/b17f4305-741a...,business communication,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications
2,http://data.europa.eu/esco/skill/d5145a9a-602e...,leadership principles,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications
3,http://data.europa.eu/esco/skill/15d76317-c71a...,communication,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications
4,http://data.europa.eu/esco/skill/a6e1e0a0-07f8...,hand gestures,http://data.europa.eu/esco/isced-f/00,generic programmes and qualifications


In [31]:
skill_levels_wide = skill_levels_wide.drop(columns=['skill_label'])

### 3.04 Merge skill categories to skills_df

In [32]:
# Rename columns for consistency
skills_df = skills_df.rename(columns={
    'skillLabel': 'skill_label',
    'skillType': 'skill_type',
    'skillUri': 'skill_uri'
})

# Merge skill_relations onto skills_df using skill_code
skills_df = skills_df.merge(skill_levels_wide, on='skill_uri', how='left')

print(f"Merged skill_relations onto skills_df")
print(f"Final shape: {skills_df.shape}")

# Count missing category labels
missing_cat_1 = skills_df['skill_category_uri'].isna().sum()

print(f"\nMissing skill_category_uri: {missing_cat_1} ({missing_cat_1/len(skills_df)*100:.1f}%)")

skills_df.head()

Merged skill_relations onto skills_df
Final shape: (99274, 27)

Missing skill_category_uri: 14 (0.0%)


,esco_id,esco_label,esco_description,group_code,group_label_en,division_code,division_label_en,section_code,section_label_en,pad_occupations,...,skill_uri,skill_label,occupation_uuid,skill_code,esco_num,relevant,top_five,project_id,skill_category_uri,skill_category_label
0,0561328b-875b-4ae2-9ba1-9af9049aef01,procurement category specialist,Procurement category specialists are experts i...,702.0,70.2 Business and other management consultancy...,70.0,70 Activities of head offices and management c...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...","""procurement specialist"", ""senior procurement ...",...,http://data.europa.eu/esco/skill/23ac233d-84ad...,monitor developments in field of expertise,0561328b-875b-4ae2-9ba1-9af9049aef01,23ac233d-84ad-4517-b0f5-8ca19ba2614e,1,True,False,P177099,http://data.europa.eu/esco/skill/0a2d70ee-d435...,information skills
1,0561328b-875b-4ae2-9ba1-9af9049aef01,procurement category specialist,Procurement category specialists are experts i...,702.0,70.2 Business and other management consultancy...,70.0,70 Activities of head offices and management c...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...","""procurement specialist"", ""senior procurement ...",...,http://data.europa.eu/esco/skill/305dc408-68a6...,develop performance orientation in public admi...,0561328b-875b-4ae2-9ba1-9af9049aef01,305dc408-68a6-4f63-9ee6-d0b26c8abad0,1,True,False,P177099,http://data.europa.eu/esco/skill/869fc2ce-478f...,management skills
2,0561328b-875b-4ae2-9ba1-9af9049aef01,procurement category specialist,Procurement category specialists are experts i...,702.0,70.2 Business and other management consultancy...,70.0,70 Activities of head offices and management c...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...","""procurement specialist"", ""senior procurement ...",...,http://data.europa.eu/esco/skill/31599ac3-e03c...,draft procurement technical specifications,0561328b-875b-4ae2-9ba1-9af9049aef01,31599ac3-e03c-42f8-8f08-b549af8234f2,1,True,True,P177099,http://data.europa.eu/esco/skill/869fc2ce-478f...,management skills
3,0561328b-875b-4ae2-9ba1-9af9049aef01,procurement category specialist,Procurement category specialists are experts i...,702.0,70.2 Business and other management consultancy...,70.0,70 Activities of head offices and management c...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...","""procurement specialist"", ""senior procurement ...",...,http://data.europa.eu/esco/skill/3bc08c26-2f98...,procurement lifecycle,0561328b-875b-4ae2-9ba1-9af9049aef01,3bc08c26-2f98-4730-bbb6-ec4a8566e3cf,1,True,True,P177099,http://data.europa.eu/esco/isced-f/04,"business, administration and law"
4,0561328b-875b-4ae2-9ba1-9af9049aef01,procurement category specialist,Procurement category specialists are experts i...,702.0,70.2 Business and other management consultancy...,70.0,70 Activities of head offices and management c...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...","""procurement specialist"", ""senior procurement ...",...,http://data.europa.eu/esco/skill/3cd35f5d-ce6d...,maintain relationship with suppliers,0561328b-875b-4ae2-9ba1-9af9049aef01,3cd35f5d-ce6d-4f14-9a09-53d7a28d834c,1,True,False,P177099,http://data.europa.eu/esco/skill/dc06de9f-dd3a...,"communication, collaboration and creativity"


### 3.04 Select and reorder columns

In [33]:
# Select and reorder columns
columns_to_keep = [
    'project_id', 'esco_id', 'skill_code', 'skill_label', 'skill_type', 'skill_category_label', 'top_five'
]

skills_df = skills_df[columns_to_keep]

print(f"Selected and reordered to {len(columns_to_keep)} columns")
print(f"Final shape: {skills_df.shape}")
skills_df.head()

Selected and reordered to 7 columns
Final shape: (99274, 7)


,project_id,esco_id,skill_code,skill_label,skill_type,skill_category_label,top_five
0,P177099,0561328b-875b-4ae2-9ba1-9af9049aef01,23ac233d-84ad-4517-b0f5-8ca19ba2614e,monitor developments in field of expertise,skill/competence,information skills,False
1,P177099,0561328b-875b-4ae2-9ba1-9af9049aef01,305dc408-68a6-4f63-9ee6-d0b26c8abad0,develop performance orientation in public admi...,skill/competence,management skills,False
2,P177099,0561328b-875b-4ae2-9ba1-9af9049aef01,31599ac3-e03c-42f8-8f08-b549af8234f2,draft procurement technical specifications,skill/competence,management skills,True
3,P177099,0561328b-875b-4ae2-9ba1-9af9049aef01,3bc08c26-2f98-4730-bbb6-ec4a8566e3cf,procurement lifecycle,knowledge,"business, administration and law",True
4,P177099,0561328b-875b-4ae2-9ba1-9af9049aef01,3cd35f5d-ce6d-4f14-9a09-53d7a28d834c,maintain relationship with suppliers,skill/competence,"communication, collaboration and creativity",False


### 3.05 Save skill level data

In [34]:
# Save to CSV
output_file = GOLD_DIR / "skill_level_data.csv"
skills_df.to_csv(output_file, index=False)

print(f"✓ Saved {len(skills_df)} skill records to {output_file}")
print(f"  Columns: {len(skills_df.columns)}")
print(f"  File size: {output_file.stat().st_size / 1024:.1f} KB")

✓ Saved 99274 skill records to ../data/gold/skill_level_data.csv
  Columns: 7
  File size: 15690.1 KB


## 4. Save project_occupation_data

In [35]:
# Re-read gold data
project_level_file = GOLD_DIR / "project_level_data.csv"
occupation_level_file = GOLD_DIR / "occupation_level_data.csv"

project_level_df = pd.read_csv(project_level_file)
occupation_level_df = pd.read_csv(occupation_level_file)

# Merge project_level_data onto occupation_level_data on project_id
merged_df = occupation_level_df.merge(project_level_df, on="project_id", how="left", suffixes=('', '_proj'))

# Get project_level columns, dropping 'long_summary'
project_level_cols = [col for col in project_level_df.columns if col not in ["long_summary", "borrower_2","disclosure_date", "effective_date", "implementing_agency", "last_stage_reached", "last_update_date"]]

# Get occupation_level columns not in project_level (to avoid duplicates)
occupation_only_cols = [col for col in occupation_level_df.columns if col not in project_level_cols]

# Order columns: project_level first (without long_summary), then occupation_level
ordered_cols = project_level_cols + occupation_only_cols
merged_df = merged_df[ordered_cols]

print(f"Merged shape: {merged_df.shape}")
merged_df.head()

Merged shape: (4933, 24)


,project_id,project_title,geography,short_summary,closing_date,total_project_cost_1,status,team_leader,region,environmental_and_social_risk,...,industry_cat_label,industry_division_code,industry_division_label,onet_job_zone,onet_job_zone_label,onet_job_zone_est,pad_job_titles,pad_activities,pad_skills,pad_quotes
0,P166796,Mali - Electricity Sector Improvement Project,Mali,Supports electricity reform in Mali by financi...,"July 31, 2026",US$ 152.10 million,Active,Abdou Mbaye,Western and Central Africa,Not Applicable,...,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...",69.0,69 Legal and accounting activities,4.0,4: Considerable Preparation Needed,llm,"""accounts payable specialist""","""Process and schedule payments to suppliers to...","""supplier payment processing"", ""cash flow plan...",ANNEX 5: FINANCIAL SITUATION OF THE POWER SECT...
1,P166796,Mali - Electricity Sector Improvement Project,Mali,Supports electricity reform in Mali by financi...,"July 31, 2026",US$ 152.10 million,Active,Abdou Mbaye,Western and Central Africa,Not Applicable,...,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...",70.0,70 Activities of head offices and management c...,2.0,2: Some Preparation Needed,minimum,"""procurement specialist""","""Revise and implement efficient fuel procureme...","""competitive tendering"", ""supplier evaluation""...",ANNEX 5: FINANCIAL SITUATION OF THE POWER SECT...
2,P166796,Mali - Electricity Sector Improvement Project,Mali,Supports electricity reform in Mali by financi...,"July 31, 2026",US$ 152.10 million,Active,Abdou Mbaye,Western and Central Africa,Not Applicable,...,T OTHER SERVICE ACTIVITIES,95.0,"95 Repair and maintenance of computers, person...",2.0,2: Some Preparation Needed,minimum,"""call center technician""","""Install the customer call center as part of c...","""telephony system installation"", ""call center ...","VII. RESULTS FRAMEWORK AND MONITORING: ""The ca..."
3,P166796,Mali - Electricity Sector Improvement Project,Mali,Supports electricity reform in Mali by financi...,"July 31, 2026",US$ 152.10 million,Active,Abdou Mbaye,Western and Central Africa,Not Applicable,...,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...",71.0,71 Architectural and engineering activities; t...,3.0,3: Medium Preparation Needed,llm,"""substation technician""","""Perform hands-on rehabilitation, installation...","""switchgear installation"", ""protection relay t...","ANNEX 4: ECONOMIC AND FINANCIAL ANALYSIS: ""The..."
4,P166796,Mali - Electricity Sector Improvement Project,Mali,Supports electricity reform in Mali by financi...,"July 31, 2026",US$ 152.10 million,Active,Abdou Mbaye,Western and Central Africa,Not Applicable,...,Q EDUCATION,85.0,85 Education,4.0,4: Considerable Preparation Needed,minimum,"""training specialist""","""Deliver capacity strengthening activities to ...","""training needs assessment"", ""curriculum devel...","ANNEX 4: ECONOMIC AND FINANCIAL ANALYSIS: ""The..."


In [36]:
# Save merged_df to gold/project_occupation_data.csv
output_file = GOLD_DIR / "project_occupation_data.csv"
merged_df.to_csv(output_file, index=False)

print(f"✓ Saved {len(merged_df)} rows to {output_file}")
print(f"  Columns: {len(merged_df.columns)}")
print(f"  File size: {output_file.stat().st_size / 1024:.1f} KB")

✓ Saved 4933 rows to ../data/gold/project_occupation_data.csv
  Columns: 24
  File size: 11027.3 KB


## 5. Save project_occupation_skills_data

In [37]:
# Read in skill level data
skill_level_file = GOLD_DIR / "skill_level_data.csv"
skill_level_df = pd.read_csv(skill_level_file)
print(skill_level_df.columns)

Index(['project_id', 'esco_id', 'skill_code', 'skill_label', 'skill_type',
       'skill_category_label', 'top_five'],
      dtype='object')


In [38]:
# Merge merged_df with skill_level_df on project_id and esco_id
final_skills_df = merged_df.merge(skill_level_df, on=["project_id", "esco_id"], how="left")

In [39]:
# Order columns: merged_df columns first, then skill_level_df columns (excluding project_id and esco_id to avoid duplicates)
skill_level_cols = [col for col in skill_level_df.columns if col not in ["project_id", "esco_id"]]
final_columns = list(merged_df.columns) + skill_level_cols
final_skills_df = final_skills_df[final_columns]
final_skills_df.head()

,project_id,project_title,geography,short_summary,closing_date,total_project_cost_1,status,team_leader,region,environmental_and_social_risk,...,onet_job_zone_est,pad_job_titles,pad_activities,pad_skills,pad_quotes,skill_code,skill_label,skill_type,skill_category_label,top_five
0,P166796,Mali - Electricity Sector Improvement Project,Mali,Supports electricity reform in Mali by financi...,"July 31, 2026",US$ 152.10 million,Active,Abdou Mbaye,Western and Central Africa,Not Applicable,...,llm,"""accounts payable specialist""","""Process and schedule payments to suppliers to...","""supplier payment processing"", ""cash flow plan...",ANNEX 5: FINANCIAL SITUATION OF THE POWER SECT...,08371fe9-c658-4f6a-966f-71dbb0ab683d,accounting department processes,knowledge,"business, administration and law",False
1,P166796,Mali - Electricity Sector Improvement Project,Mali,Supports electricity reform in Mali by financi...,"July 31, 2026",US$ 152.10 million,Active,Abdou Mbaye,Western and Central Africa,Not Applicable,...,llm,"""accounts payable specialist""","""Process and schedule payments to suppliers to...","""supplier payment processing"", ""cash flow plan...",ANNEX 5: FINANCIAL SITUATION OF THE POWER SECT...,147170ba-0141-4337-89af-b4513d745afe,support development of annual budget,skill/competence,management skills,False
2,P166796,Mali - Electricity Sector Improvement Project,Mali,Supports electricity reform in Mali by financi...,"July 31, 2026",US$ 152.10 million,Active,Abdou Mbaye,Western and Central Africa,Not Applicable,...,llm,"""accounts payable specialist""","""Process and schedule payments to suppliers to...","""supplier payment processing"", ""cash flow plan...",ANNEX 5: FINANCIAL SITUATION OF THE POWER SECT...,21c5790c-0930-4d74-b3b0-84caf5af12ea,manage budgets,skill/competence,management skills,False
3,P166796,Mali - Electricity Sector Improvement Project,Mali,Supports electricity reform in Mali by financi...,"July 31, 2026",US$ 152.10 million,Active,Abdou Mbaye,Western and Central Africa,Not Applicable,...,llm,"""accounts payable specialist""","""Process and schedule payments to suppliers to...","""supplier payment processing"", ""cash flow plan...",ANNEX 5: FINANCIAL SITUATION OF THE POWER SECT...,2313b15d-a9a7-4830-9bcc-fd7df79d842b,enforce financial policies,skill/competence,management skills,False
4,P166796,Mali - Electricity Sector Improvement Project,Mali,Supports electricity reform in Mali by financi...,"July 31, 2026",US$ 152.10 million,Active,Abdou Mbaye,Western and Central Africa,Not Applicable,...,llm,"""accounts payable specialist""","""Process and schedule payments to suppliers to...","""supplier payment processing"", ""cash flow plan...",ANNEX 5: FINANCIAL SITUATION OF THE POWER SECT...,2ada2f9d-77f7-4ba0-9096-89ae285a65c5,monitor financial accounts,skill/competence,information skills,True


In [40]:
# Save final_skills_df to gold/project_occupation_skill_data.csv
output_file = GOLD_DIR / "project_occupation_skill_data.csv"
final_skills_df.to_csv(output_file, index=False)

## 6. Files with fewer columns for visualization

In [43]:
# Create project_occupation_data_viz.csv with selected columns
viz_occupation_cols = [
    "project_id",
    "project_title",
    "short_summary",
    "esco_id",
    "occupation_esco",
    "industry_cat_label",
    "onet_job_zone",
    "onet_job_zone_label",
    "pad_activities"
]

# Verify all columns exist in merged_df
missing_cols = [col for col in viz_occupation_cols if col not in merged_df.columns]
if missing_cols:
    print(f"Warning: Missing columns in merged_df: {missing_cols}")
else:
    project_occ_viz_df = merged_df[viz_occupation_cols].copy()
    
    # Only keep short_summary for the first row of each project_id
    project_occ_viz_df['short_summary'] = project_occ_viz_df.groupby('project_id')['short_summary'].transform(
        lambda x: [x.iloc[0]] + [''] * (len(x) - 1)
    )
    
    output_file = GOLD_DIR / "project_occupation_data_viz.csv"
    project_occ_viz_df.to_csv(output_file, index=False)
    
    print(f"✓ Saved {len(project_occ_viz_df)} rows to {output_file}")
    print(f"  Columns: {len(project_occ_viz_df.columns)}")
    print(f"  File size: {output_file.stat().st_size / 1024:.1f} KB")


✓ Saved 4933 rows to ../data/gold/project_occupation_data_viz.csv
  Columns: 9
  File size: 2419.0 KB


In [44]:
# Create project_occupation_skills_data_viz.csv with selected columns
viz_skills_cols = [
    "project_title",
    "short_summary",
    "occupation_esco",
    "onet_job_zone",
    "onet_job_zone_label",
    "skill_category_label",
    "skill_label",
    "industry_cat_label",
    "top_five"
]

# Need project_id for grouping
viz_skills_cols_with_id = ["project_id"] + viz_skills_cols

# Verify all columns exist in final_skills_df
missing_cols = [col for col in viz_skills_cols_with_id if col not in final_skills_df.columns]
if missing_cols:
    print(f"Warning: Missing columns in final_skills_df: {missing_cols}")
else:
    project_occ_skills_viz_df = final_skills_df[viz_skills_cols_with_id].copy()
    
    # Only keep short_summary for the first row of each project_id
    project_occ_skills_viz_df['short_summary'] = project_occ_skills_viz_df.groupby('project_id')['short_summary'].transform(
        lambda x: [x.iloc[0]] + [''] * (len(x) - 1)
    )
    
    # Drop project_id column (not needed in final output)
    project_occ_skills_viz_df = project_occ_skills_viz_df[viz_skills_cols]
    
    output_file = GOLD_DIR / "project_occupation_skill_data_viz.csv"
    project_occ_skills_viz_df.to_csv(output_file, index=False)
    
    print(f"✓ Saved {len(project_occ_skills_viz_df)} rows to {output_file}")
    print(f"  Columns: {len(project_occ_skills_viz_df.columns)}")
    print(f"  File size: {output_file.stat().st_size / 1024:.1f} KB")


✓ Saved 99543 rows to ../data/gold/project_occupation_skill_data_viz.csv
  Columns: 9
  File size: 24255.5 KB
